# IMPORTS & SETUP

In [1]:
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Helper function to flatten multi-level column names after aggregation
def flatten_columns(df, prefix):
    df.columns = [f"{prefix}_{col[0]}_{col[1]}".strip('_') if type(col) is tuple else col for col in df.columns]
    return df

# LOAD DATASETS

In [3]:
print("Loading CSV files...")
app_train = pd.read_csv('application_train.csv')
bureau = pd.read_csv('bureau.csv')
bureau_balance = pd.read_csv('bureau_balance.csv')
prev_app = pd.read_csv('previous_application.csv')
pos_cash = pd.read_csv('POS_CASH_balance.csv')
credit_card = pd.read_csv('credit_card_balance.csv')
installments = pd.read_csv('installments_payments.csv')

Loading CSV files...


# LEVEL 2 AGGREGATION (DEEPEST TABLES)

In [4]:
print("Aggregating Level 2 tables...")

Aggregating Level 2 tables...


In [5]:
# A. bureau_balance.csv -> merge into bureau.csv
bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg({'MONTHS_BALANCE': ['min', 'max', 'size']})
bb_agg = flatten_columns(bb_agg, 'BB')
bureau = bureau.merge(bb_agg.reset_index(), on='SK_ID_BUREAU', how='left')
del bureau_balance, bb_agg
gc.collect()

0

In [6]:
# B. POS_CASH_balance.csv -> aggregate to SK_ID_CURR
pos_num = pos_cash.select_dtypes(include=[np.number]).drop(columns=['SK_ID_PREV'])
pos_agg = pos_num.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum'])
pos_agg = flatten_columns(pos_agg, 'POS')
del pos_cash, pos_num
gc.collect()

0

In [7]:
# C. credit_card_balance.csv -> aggregate to SK_ID_CURR
cc_num = credit_card.select_dtypes(include=[np.number]).drop(columns=['SK_ID_PREV'])
cc_agg = cc_num.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum'])
cc_agg = flatten_columns(cc_agg, 'CC')
del credit_card, cc_num
gc.collect()

0

In [8]:
# D. installments_payments.csv -> aggregate to SK_ID_CURR
# Feature Engineering: Days past due and Amount underpaid
installments['DPD'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['AMT_SHORTAGE'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']

In [9]:
inst_num = installments.select_dtypes(include=[np.number]).drop(columns=['SK_ID_PREV'])
inst_agg = inst_num.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum'])
inst_agg = flatten_columns(inst_agg, 'INST')
del installments, inst_num
gc.collect()

0

# LEVEL 1 AGGREGATION (DIRECT AUX TABLES)

In [10]:
print("Aggregating Level 1 tables...")

Aggregating Level 1 tables...


In [11]:
# A. bureau.csv (now containing bureau_balance) -> aggregate to SK_ID_CURR
bureau_num = bureau.select_dtypes(include=[np.number]).drop(columns=['SK_ID_BUREAU'])
bureau_agg = bureau_num.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum'])
bureau_agg = flatten_columns(bureau_agg, 'BUREAU')
del bureau, bureau_num
gc.collect()

0

In [12]:
# B. previous_application.csv -> aggregate to SK_ID_CURR
prev_num = prev_app.select_dtypes(include=[np.number]).drop(columns=['SK_ID_PREV'])
prev_agg = prev_num.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum'])
prev_agg = flatten_columns(prev_agg, 'PREV')
del prev_app, prev_num
gc.collect()

0

# THE FINAL MASTER MERGE

In [13]:
print("Merging all aggregated features into application_train...")

Merging all aggregated features into application_train...


In [14]:
# Merge sequentially to monitor memory
app_train = app_train.merge(bureau_agg.reset_index(), on='SK_ID_CURR', how='left')
app_train = app_train.merge(prev_agg.reset_index(), on='SK_ID_CURR', how='left')
app_train = app_train.merge(pos_agg.reset_index(), on='SK_ID_CURR', how='left')
app_train = app_train.merge(cc_agg.reset_index(), on='SK_ID_CURR', how='left')
app_train = app_train.merge(inst_agg.reset_index(), on='SK_ID_CURR', how='left')

In [15]:
# Clean up memory
del bureau_agg, prev_agg, pos_agg, cc_agg, inst_agg
gc.collect()

0

In [16]:
print(f"Master dataset ready! Final shape: {app_train.shape}")

Master dataset ready! Final shape: (307511, 390)


# DOMAIN FEATURE ENGINEERING

In [17]:
print("Engineering domain features...")
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import matplotlib.pyplot as plt

Engineering domain features...


In [18]:
# Create financial ratios to boost model performance
app_train['CREDIT_INCOME_PERCENT'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']
app_train['ANNUITY_INCOME_PERCENT'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
app_train['CREDIT_TERM'] = app_train['AMT_ANNUITY'] / app_train['AMT_CREDIT']
app_train['DAYS_EMPLOYED_PERCENT'] = app_train['DAYS_EMPLOYED'] / app_train['DAYS_BIRTH']

# DATA PREPROCESSING

In [19]:
print("Preprocessing data for modeling...")

Preprocessing data for modeling...


In [20]:
# Separate target and features
y = app_train['TARGET']
X = app_train.drop(columns=['TARGET', 'SK_ID_CURR'])

In [21]:
# Label Encode categorical variables 
le = LabelEncoder()
cat_cols = X.select_dtypes(include=['object', 'str']).columns
for col in cat_cols:
    X[col] = X[col].astype(str)
    X[col] = le.fit_transform(X[col])

In [22]:
# Replace any infinity values created by division with NaN
X.replace([np.inf, -np.inf], np.nan, inplace=True)

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,INST_DPD_mean,INST_DPD_sum,INST_AMT_SHORTAGE_min,INST_AMT_SHORTAGE_max,INST_AMT_SHORTAGE_mean,INST_AMT_SHORTAGE_sum,CREDIT_INCOME_PERCENT,ANNUITY_INCOME_PERCENT,CREDIT_TERM,DAYS_EMPLOYED_PERCENT
0,0,1,0,1,0,202500.0,406597.5,24700.5,351000.0,6,...,-20.421053,-388.0,0.000,0.000,0.000000,0.000,2.007889,0.121978,0.060749,0.067329
1,0,0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,1,...,-7.160000,-179.0,0.000,0.000,0.000000,0.000,4.790750,0.132217,0.027598,0.070862
2,1,1,1,1,0,67500.0,135000.0,6750.0,135000.0,6,...,-7.666667,-23.0,0.000,0.000,0.000000,0.000,2.000000,0.100000,0.050000,0.011814
3,0,0,0,1,0,135000.0,312682.5,29686.5,297000.0,6,...,-19.375000,-310.0,0.000,0.000,0.000000,0.000,2.316167,0.219900,0.094941,0.159905
4,0,1,0,1,0,121500.0,513000.0,21865.5,513000.0,6,...,-3.636364,-240.0,0.000,22655.655,452.384318,29857.365,4.222222,0.179963,0.042623,0.152418
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,0,1,0,0,0,157500.0,254700.0,27558.0,225000.0,6,...,-36.285714,-254.0,0.000,0.000,0.000000,0.000,1.617143,0.174971,0.108198,0.025303
307507,0,0,0,1,0,72000.0,269550.0,12001.5,225000.0,6,...,-2.833333,-17.0,0.000,0.000,0.000000,0.000,3.743750,0.166687,0.044524,-17.580890
307508,0,0,0,1,0,153000.0,677664.0,29979.0,585000.0,6,...,-14.500000,-203.0,0.000,3945.825,283.792500,3973.095,4.429176,0.195941,0.044239,0.529266
307509,0,0,0,1,0,171000.0,370107.0,20205.0,319500.0,6,...,-19.000000,-361.0,0.000,0.000,0.000000,0.000,2.164368,0.118158,0.054592,0.400134


In [23]:
# Train-Test Split (Stratified to handle the ~8% default imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [24]:
# Impute missing values (Required for Logistic Regression & Random Forest)
print("Imputing missing values for baseline models (this may take a moment)...")
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

Imputing missing values for baseline models (this may take a moment)...


# PREDICTIVE MODELING (TASK 2)

In [25]:
print("Training Baseline: Logistic Regression...")
lr_model = LogisticRegression(class_weight='balanced', max_iter=200)
lr_model.fit(X_train_imputed, y_train)
lr_preds = lr_model.predict_proba(X_test_imputed)[:, 1]
lr_auc = roc_auc_score(y_test, lr_preds)

Training Baseline: Logistic Regression...


In [26]:
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=50, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_imputed, y_train)
rf_preds = rf_model.predict_proba(X_test_imputed)[:, 1]
rf_auc = roc_auc_score(y_test, rf_preds)

Training Random Forest Classifier...


In [27]:
print("Training LightGBM Classifier...")
# LightGBM handles NaNs natively, so we can pass the un-imputed, un-scaled data directly
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'is_unbalance': True, # Crucial parameter for highly imbalanced target
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

Training LightGBM Classifier...


In [28]:
lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=300)
lgb_preds = lgb_model.predict(X_test)
lgb_auc = roc_auc_score(y_test, lgb_preds)

# MODEL COMPARISON REPORT

In [29]:
print("\n" + "="*40)
print("       MODEL COMPARISON REPORT")
print("="*40)
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'LightGBM'],
    'ROC-AUC Score': [lr_auc, rf_auc, lgb_auc]
})


       MODEL COMPARISON REPORT


In [30]:
print(results.sort_values(by='ROC-AUC Score', ascending=False).to_markdown(index=False))
print("="*40)
print("\nRecommendation for Production: LightGBM")
print("Reasoning: LightGBM natively handles the massive amount of missing data in the dataset, effectively manages the heavy class imbalance using the 'is_unbalance' parameter, and typically yields the highest ROC-AUC score while executing much faster than Random Forest.")

| Model               |   ROC-AUC Score |
|:--------------------|----------------:|
| LightGBM            |        0.78763  |
| Random Forest       |        0.751723 |
| Logistic Regression |        0.66888  |

Recommendation for Production: LightGBM
Reasoning: LightGBM natively handles the massive amount of missing data in the dataset, effectively manages the heavy class imbalance using the 'is_unbalance' parameter, and typically yields the highest ROC-AUC score while executing much faster than Random Forest.


# Comprehensive Project Report: Home Loan Default
* **PredictionReference Document:** PRCP-1006-HomeLoanDef.docx
* **Domain:** Banking
# 1. Problem Statement
Financial institutions need to accurately assess the risk of a client defaulting on a loan to make informed lending decisions. The objective of this project is two-fold:
* **Task 1:** Prepare a complete data analysis report on the given historical loan data to understand applicant profiles.
* **Task 2:** Create a predictive machine learning model to identify the factors and customer segments that are eligible for taking a loan (i.e., identifying potential non-defaulters vs. defaulters).
# 2. Task 1: Data Analysis Report (Summary)
The project utilizes a complex, relational database consisting of 7 interconnected datasets, with application_train.csv acting as the primary hub containing the TARGET variable (1: Defaulter; 0: Not Defaulter).
## Key Insights from Exploratory Data Analysis (EDA):
 * **Target Imbalance:** The dataset is highly imbalanced, with defaulters making up approximately 8% of the total applications. This required specialized sampling and metric evaluation strategies later in the pipeline.
 * **Applicant Financial Health:** Derived domain features such as CREDIT_INCOME_PERCENT (Loan Amount / Total Income) and ANNUITY_INCOME_PERCENT proved to be critical indicators of a client's financial strain.
 * **Historical Behavior:**  A client's past credit history (captured in bureau.csv and previous_application.csv) and their installment payment behaviors (like AMT_SHORTAGE and Days Past Due derived from installments_payments.csv) are strong indicators of future repayment capabilities.
# 3. Report on Challenges Faced
Processing this dataset presented several significant challenges, requiring specific data engineering and machine learning techniques to overcome:
## Challenge 1: Complex Relational Data and Memory Constraints
* **Issue:** The data was spread across 7 different files with millions of rows detailing monthly balances and historical interactions. Directly merging them would exceed memory limits and create duplicate records.
* **Technique Used:** Implemented a Two-Level Aggregation Strategy. Deepest tables (e.g., bureau_balance) were first aggregated to their intermediate IDs (SK_ID_BUREAU), merged upward, and then all auxiliary tables were grouped by the applicant ID (SK_ID_CURR) using statistical functions (min, max, mean, sum). Sequential merging with explicit memory cleanup (gc.collect()) was used to prevent kernel crashes, resulting in a single flattened master dataset of 307,511 rows and 390 features.
## Challenge 2: Massive Volume of Missing Data
* **Issue:** Many aggregated columns (especially from historical credit card or POS balances) contained substantial missing values because not all applicants had prior histories in those categories.
* **Technique Used:** For baseline models like Logistic Regression and Random Forest, Median Imputation was applied to prevent data leakage and handle skewness. For the final model (LightGBM), the algorithm's native NaN handling was utilized, which treats missing values as a distinct feature distribution, preserving the informational value of the "missingness" itself.
## Challenge 3: Severe Class Imbalance
* **Issue:** With an ~8% default rate, a standard model would bias heavily toward predicting the majority class (Not Defaulter), achieving high accuracy but failing at the primary business objective.
* **Technique Used:** Accuracy was discarded as a metric in favor of ROC-AUC (Area Under the Receiver Operating Characteristic Curve). During training, a stratified train-test split was used, and class weights were adjusted (class_weight='balanced' in Scikit-Learn; is_unbalance=True in LightGBM) to heavily penalize the model for missing actual defaulters.
# 4. Task 2: Predictive Modeling and Model Comparison Report
Three distinct machine learning models were trained on the engineered dataset to predict the TARGET variable.
* **Evaluation Metric:** ROC-AUC                                                                                     
| Model | ROC-AUC Score | Key Hyperparameters / Setup |
|---|---|---|
| LightGBM | 0.78763 | learning_rate=0.05, num_leaves=31, is_unbalance=True, Native Missing Value Handling |
| Random Forest| 0.75172 | n_estimators=50, max_depth=10, class_weight='balanced',Median Imputed Data |
| Logistic Regression | 0.66888 | max_iter=200, class_weight='balanced', Median Imputed Data |                         
# Production Recommendation
## Recommended Model: LightGBM
* **Reasoning for Production:**
1. * **Superior Predictive Power:** It achieved the highest ROC-AUC score (0.787), indicating the best ability to distinguish between defaulters and non-defaulters.
2. * **Robustness to Missing Data:** It natively handles the sparsity created by flattening 7 relational tables, saving the computational overhead of imputing hundreds of features in a production environment.
3. * **Imbalance Handling:** The is_unbalance parameter mathematically corrects for the 92/8 class distribution highly effectively.
4. * **Operational Efficiency:** Gradient boosting frameworks like LightGBM train and predict significantly faster than extensive Random Forest ensembles, making it highly suitable for real-time or batch loan eligibility scoring.